## Dataset Description

The IMDb movie reviews dataset contains textual reviews labeled as either positive or
negative sentiment. The dataset is commonly used for binary text classification and
serves as a benchmark for natural language processing tasks.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## Target Variable

The target variable is `sentiment`, which indicates whether a movie review expresses
positive or negative sentiment. This is a binary classification problem.


In [ ]:
df['sentiment'].value_counts()

In [ ]:
df['review_length']=df['review'].apply(lambda x : len(x.split()))
df['review_length'].describe()

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df['review_length'],bins=50)
plt.title('Distribution of Review Lengths')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.show()

## Initial Observations

The dataset contains an equal number of positive and negative reviews, making it well
suited for binary classification. Review lengths vary significantly, indicating the
need for padding and truncation during preprocessing for deep learning models.


## Text Preprocessing

In this step, raw text reviews are converted into numerical sequences using tokenization.
Sequences are padded to a fixed length to ensure uniform input size for deep learning
models. This preprocessing is essential for training neural networks on textual data.


In [ ]:
df['label']=df['sentiment'].map({'positive':1,'negative':0})
df[['sentiment','label']].head()

In [ ]:
from sklearn.model_selection import train_test_split
x_train_text,x_test_text,y_train,y_test=train_test_split(
    df['review'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
max_words=20000
tokenizer=Tokenizer(num_words=max_words,oov_token='<OOV>')
tokenizer.fit_on_texts(x_train_text)

In [ ]:
x_train_seq=tokenizer.texts_to_sequences(x_train_text)
x_test_seq=tokenizer.texts_to_sequences(x_test_text)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_length=250
x_train_pad = pad_sequences(x_train_seq, maxlen=250, padding='post', truncating='post')
x_test_pad = pad_sequences(x_test_seq, maxlen=250, padding='post', truncating='post')

x_train_pad.shape,x_test_pad.shape

In [ ]:
word_index=tokenizer.word_index
len(word_index)

### Preprocessing Summary

Text reviews were tokenized and converted into numerical sequences using a fixed
vocabulary size. Sequences were padded and truncated to a uniform length based on the
review length distribution. This preprocessing ensures consistent input dimensions and
efficient learning for deep learning models.


## Day 1 Conclusion

The IMDb movie reviews dataset was successfully explored to understand its structure,
class balance, and textual characteristics. Analysis of review lengths revealed a
right-skewed distribution, highlighting the need for padding and truncation when working
with sequence-based models.

The text data was then preprocessed by encoding sentiment labels, tokenizing reviews,
and converting them into padded numerical sequences using a fixed vocabulary size.
A train–test split was applied to ensure balanced class representation. These steps
provide a clean and well-structured foundation for training deep learning models for
sentiment classification in the next stage.


## LSTM Model Training

In this step, a Long Short-Term Memory (LSTM) neural network is built and trained to
classify movie reviews based on sentiment. LSTM models are well suited for sequential
text data as they can capture contextual dependencies across words.


In [ ]:
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
from tensorflow.keras.layers import GlobalMaxPooling1D

model = Sequential([
    Embedding(
        input_dim=max_words,
        output_dim=128,
        input_length=max_length
    ),
    
    Bidirectional(LSTM(128, return_sequences=True)),
    GlobalMaxPooling1D(),
    
    Dense(64, activation='relu'),
    Dropout(0.3),
    
    Dense(1, activation='sigmoid')
])


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    x_train_pad, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
test_loss, test_accuracy = model.evaluate(x_test_pad, y_test)
print("Test Accuracy:", test_accuracy)


### Model Improvement Results

Replacing the standard LSTM with a Bidirectional LSTM followed by
GlobalMaxPooling1D significantly improved performance. This architecture allows
the model to capture contextual information from both directions and extract the
strongest sentiment-related features across the review. The updated model achieved
a test accuracy of approximately 88.8%, representing a clear improvement over the
baseline LSTM.


## Model Evaluation and Analysis

In this step, the trained Bidirectional LSTM model is evaluated in detail using
classification metrics and a confusion matrix. This analysis helps understand
model strengths, weaknesses, and potential areas for improvement.


In [ ]:
import numpy as np

y_pred_prob=model.predict(x_test_pad)
y_pred=(y_pred_prob > 0.5).astype(int).flatten()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm=confusion_matrix(y_test,y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

### Confusion Matrix Insight

The confusion matrix shows how well the model distinguishes between positive and
negative reviews. Most predictions fall along the diagonal, indicating strong
classification performance. Misclassifications typically occur in reviews containing
mixed or subtle sentiment expressions.


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history.history['accuracy'],label='Training Accuracy')
plt.plot(history.history['val_accuracy'],label='Valdation Accuracy')
plt.legend()
plt.title('Training vs Validation Accuracy')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title("Training vs Validation Loss")
plt.show()


### Learning Curve Interpretation

The training and validation curves show stable convergence with minimal overfitting.
Early stopping helped prevent excessive training once validation performance plateaued.
The model generalizes well to unseen data.


In [ ]:
sample_index = 10

print("Review:")
print(x_test_text.iloc[sample_index])
print("\nActual Sentiment:", y_test.iloc[sample_index])
print("Predicted Sentiment:", y_pred[sample_index])


## Final Conclusion

This project implemented an end-to-end deep learning pipeline for movie review
sentiment classification. Starting from raw text, reviews were tokenized, padded,
and transformed into numerical sequences suitable for neural networks.

A Bidirectional LSTM model with global max pooling was trained and evaluated,
achieving approximately 88.8% test accuracy. The results demonstrate the effectiveness
of recurrent neural networks for natural language understanding tasks.

This notebook highlights practical NLP preprocessing, sequence modeling, and
evaluation techniques in deep learning.
